---
## STAGE 1 — Setup & Data Loading

In [ ]:
# Import All Libraries

import requests
import feedparser
import pandas as pd
import json
import time
import warnings
from datetime import datetime
from bs4 import BeautifulSoup
from transformers import pipeline
from groq import Groq

warnings.filterwarnings('ignore')   # suppress SSL warnings from requests

print('All libraries imported successfully')
print(f'   Run started at: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

In [ ]:
# Load Companies from CSV

df_members = pd.read_csv('combined_companies.csv')

print(f'Loaded {len(df_members)} companies')
print(f'\n--- Columns available ---')
print(df_members.columns.tolist())
print(f'\n--- Sectors covered ---')
print(df_members['sector'].value_counts().to_string())
print(f'\n--- First 5 rows ---')
print(df_members[['company_name', 'county', 'sector', 'group']].head().to_string(index=False))

# Convert to list of dicts — used in all loops below
companies = df_members.to_dict('records')
print(f'\n companies list ready ({len(companies)} entries)')

---
## STAGE 2 — Talk Score (Hybrid: LLM + FinBERT on Claims)

**What is the Talk Score?**

Imagine sending each company's website to an expert ESG auditor. The auditor reads it and gives you two independent verdicts:
- **LLM verdict** — *How specific and credible are these claims?* (quality audit)
-**FinBERT verdict** — *How positively are these claims written?* (tone audit)

We combine both into one **Hybrid Talk Score (0–10)**:
```
Hybrid Talk = (LLM quality × 0.6) + (FinBERT claim tone × 0.4)
```

In [ ]:
# Load FinBERT Model

# FinBERT is a version of BERT (a language model) that was
# specially trained on financial and ESG text.

# First run: downloads ~500MB model (one-time only)
# After that: loads from local cache in seconds

print('⏳ Loading FinBERT... (may take 1–2 mins on first run)')

finbert = pipeline(
    task='text-classification',
    model='ProsusAI/finbert',   # ESG-trained model on HuggingFace
    top_k=None                  # return ALL 3 scores: positive / neutral / negative
)

print(' FinBERT loaded!')

# --- Quick sanity check ---
test = finbert('The company reduced carbon emissions by 40% and won a sustainability award.')[0]
print(f'\n Quick test result:')
for item in test:
    print(f'   {item["label"]:<10}: {item["score"]:.4f}')

In [ ]:
# Define Core FinBERT Sentiment Function
# This reusable function takes a list of text snippets,
# runs FinBERT on each one, and returns the average scores.


def analyse_sentiment(texts):
    """
    Input : list of strings  e.g. ['We reduced emissions', 'Award winner']
    Output: dict             e.g. {'positive': 0.72, 'neutral': 0.21, 'negative': 0.07}
    """
    if not texts:
        return {'positive': 0.0, 'neutral': 1.0, 'negative': 0.0}

    pos_scores, neu_scores, neg_scores = [], [], []

    for text in texts:
        text = str(text)[:512]   # FinBERT has a 512-token limit
        try:
            result = finbert(text)[0]   # returns list of 3 dicts
            for item in result:
                if   item['label'] == 'positive': pos_scores.append(item['score'])
                elif item['label'] == 'neutral':  neu_scores.append(item['score'])
                elif item['label'] == 'negative': neg_scores.append(item['score'])
        except Exception as e:
            print(f'   Skipped one text: {str(e)[:50]}')
            continue

    # Average across all texts — like averaging exam scores
    avg_pos = sum(pos_scores) / len(pos_scores) if pos_scores else 0.0
    avg_neu = sum(neu_scores) / len(neu_scores) if neu_scores else 0.0
    avg_neg = sum(neg_scores) / len(neg_scores) if neg_scores else 0.0

    return {
        'positive': round(avg_pos, 4),
        'neutral' : round(avg_neu, 4),
        'negative': round(avg_neg, 4)
    }


def scores_to_sentiment_score(scores, baseline=5.0):
    """
    Converts raw FinBERT scores to a single 0–10 number.

    Formula:  5  (neutral baseline)
            + positive * 5  (pushes score UP toward 10)
            - negative * 5  (pushes score DOWN toward 0)

    ANALOGY: Like a thermometer where:
        10 = glowing positive press
         5 = neutral / mixed
         0 = very negative press
    """
    raw = baseline + (scores['positive'] * 5) - (scores['negative'] * 5)
    return round(max(0.0, min(10.0, raw)), 2)


# --- Verification tests ---
good = ['Company launches zero-waste packaging', 'Irish food brand wins sustainability award']
bad  = ['Company fined for misleading green claims', 'Environmental violations found at plant']

print('Testing on GOOD ESG news:')
g = analyse_sentiment(good)
print(f'   Scores: {g}  →  Sentiment Score: {scores_to_sentiment_score(g)}/10')

print('\n Testing on BAD ESG news:')
b = analyse_sentiment(bad)
print(f'   Scores: {b}  →  Sentiment Score: {scores_to_sentiment_score(b)}/10')

print('\n analyse_sentiment() and scores_to_sentiment_score() ready')

In [ ]:
# Setup Groq LLM Client + ESG Auditor Prompt

# The LLM acts as a senior ESG consultant who reads each
# company's website and returns a structured audit in JSON.

GROQ_API_KEY = 'REMOVED'   # 🔑 Paste your Groq key here
client = Groq(api_key=GROQ_API_KEY)

SYSTEM_PROMPT = """
You are an expert ESG auditor specialising in greenwashing detection
for Irish Food SMEs. Your job is to read company website text and
evaluate the quality of their sustainability claims.

You must respond ONLY with a valid JSON object — no preamble,
no explanation, no markdown backticks. Just raw JSON.

Evaluate claims on:
- Specificity: Vague ("we care about the environment") = LOW
               Specific ("reduced emissions 34% since 2019") = HIGH
- Credibility: No evidence = LOW | Certifications/data/targets = HIGH
- Greenwashing signals: vague language, unverified certs,
  no measurable targets, misleading comparisons

Score 1–10 where:
  1  = no ESG claims at all
  5  = generic vague claims, no evidence
  10 = specific, measurable, verified claims

Return EXACTLY this JSON:
{
  "esg_claims_found": ["claim 1", "claim 2"],
  "claim_specificity": 0,
  "claim_credibility": 0,
  "greenwashing_signals": ["signal 1"],
  "strengths": ["strength 1"],
  "auditor_note": "one sentence summary"
}
"""

print(' Groq client ready')
print(' System prompt configured')

In [ ]:
# Define Website Fetcher + LLM Auditor Functions


HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/120.0.0.0 Safari/537.36'
}

def fetch_website_text(company, max_chars=2000):
    """
    Visits a company website and extracts paragraph text.
    Returns empty string if site is inaccessible.
    """
    try:
        resp = requests.get(company['website'], headers=HEADERS, timeout=10, verify=False)
        soup = BeautifulSoup(resp.text, 'html.parser')
        paras = soup.find_all('p')
        text  = ' '.join([p.get_text(strip=True) for p in paras if len(p.get_text(strip=True)) > 40])
        return text[:max_chars] if text else ''
    except Exception:
        return ''


def llm_esg_audit(company, website_text):
    """
    Sends website text to Groq/Llama for structured ESG audit.
    Returns a dict with specificity, credibility scores, and claims.
    """
    # If website wasn't accessible, return zero scores
    if not website_text or len(website_text) < 50:
        return {
            'esg_claims_found'    : [],
            'claim_specificity'   : 1,
            'claim_credibility'   : 1,
            'greenwashing_signals': ['website not accessible'],
            'strengths'           : [],
            'auditor_note'        : 'Website could not be accessed'
        }

    user_message = f"""
Company: {company['company_name']}
Sector:  {company.get('sector', 'Food')}
Country: Ireland

Website text:
---
{website_text}
---

Provide your ESG audit as JSON only.
"""
    try:
        response = client.chat.completions.create(
            model='llama-3.1-8b-instant',
            messages=[
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user',   'content': user_message}
            ],
            temperature=0.1,
            max_tokens=500
        )
        raw = response.choices[0].message.content.strip()
        raw = raw.replace('```json', '').replace('```', '').strip()
        return json.loads(raw)

    except json.JSONDecodeError:
        return {
            'esg_claims_found'    : [],
            'claim_specificity'   : 5,
            'claim_credibility'   : 5,
            'greenwashing_signals': ['could not parse LLM response'],
            'strengths'           : [],
            'auditor_note'        : 'Parse error — manual review needed'
        }
    except Exception as e:
        return {
            'esg_claims_found'    : [],
            'claim_specificity'   : 1,
            'claim_credibility'   : 1,
            'greenwashing_signals': [f'API error: {str(e)[:50]}'],
            'strengths'           : [],
            'auditor_note'        : 'API call failed'
        }


print(' fetch_website_text() ready')
print(' llm_esg_audit() ready')

In [ ]:
# Quick Test: LLM Audit on ONE Company

test_company = companies[0]   # first company in your CSV

print(f' Fetching website: {test_company["company_name"]}')
site_text = fetch_website_text(test_company)
print(f'   Characters fetched: {len(site_text)}')

print('\n Sending to Groq for ESG audit...')
test_audit = llm_esg_audit(test_company, site_text)

print(f'\n{"-"*50}')
print(f' AUDIT RESULT — {test_company["company_name"]}')
print(f'{"-"*50}')
print(f'Claims found      : {test_audit["esg_claims_found"]}')
print(f'Specificity       : {test_audit["claim_specificity"]}/10')
print(f'Credibility       : {test_audit["claim_credibility"]}/10')
print(f'Greenwash signals : {test_audit["greenwashing_signals"]}')
print(f'Strengths         : {test_audit["strengths"]}')
print(f'Auditor note      : {test_audit["auditor_note"]}')

In [ ]:
# Run LLM + FinBERT on ALL Companies → Hybrid Talk Score
# For each company we:
#   1. Fetch website text
#   2. LLM audit  → specificity + credibility scores
#   3. FinBERT    → sentiment of the extracted ESG claims
#   4. Combine    → Hybrid Talk Score
#
# Hybrid Talk Score formula:
#   LLM quality score   = (specificity + credibility) / 2
#   FinBERT claim score = sentiment of extracted claims
#   Hybrid Talk         = (LLM × 0.6) + (FinBERT × 0.4)
#
# WHY 0.6 / 0.4?
#   We trust the LLM's structural quality assessment more,
#   but FinBERT adds a useful tone/confidence signal.
#   These weights are tunable — see dissertation Section 4.

WEIGHT_LLM     = 0.6
WEIGHT_FINBERT = 0.4

talk_results = []

print(f' Running Hybrid Talk Score pipeline on {len(companies)} companies')
print(f'   Weights: LLM={WEIGHT_LLM}, FinBERT={WEIGHT_FINBERT}')
print(f' Estimated time: ~5–8 minutes\n')

for i, company in enumerate(companies, 1):
    name = company['company_name']
    print(f'[{i}/{len(companies)}] {name}')

    site_text = fetch_website_text(company)
    print(f'    Website chars: {len(site_text)}')
    
    audit = llm_esg_audit(company, site_text)
    spec  = int(audit['claim_specificity'])
    cred  = int(audit['claim_credibility'])
    llm_quality = round((spec + cred) / 2, 2)   # simple average
    print(f'    LLM: specificity={spec}/10, credibility={cred}/10, quality={llm_quality}/10')

    
    claims_list = audit.get('esg_claims_found', [])
    if not claims_list:
        claim_sent_score = 5.0   # neutral baseline if no claims extracted
        claim_pos, claim_neg = 0.0, 0.0
    else:
        claim_scores     = analyse_sentiment(claims_list)
        claim_sent_score = scores_to_sentiment_score(claim_scores)
        claim_pos        = claim_scores['positive']
        claim_neg        = claim_scores['negative']
    print(f'    FinBERT on claims: sentiment={claim_sent_score}/10')

    
    hybrid_talk = round((llm_quality * WEIGHT_LLM) + (claim_sent_score * WEIGHT_FINBERT), 2)
    print(f'    Hybrid Talk Score: {hybrid_talk}/10')

    talk_results.append({
        'company_name'        : name,
        'sector'              : company.get('sector', 'Other'),
        'county'              : company.get('county', ''),
        'group'               : company.get('group', ''),
        'bord_bia_verified'   : company.get('bord_bia_gold_2025', False),
        # LLM sub-scores
        'claim_specificity'   : spec,
        'claim_credibility'   : cred,
        'llm_quality_score'   : llm_quality,
        'claims_found'        : ' | '.join(audit.get('esg_claims_found', [])),
        'greenwash_signals'   : ' | '.join(audit.get('greenwashing_signals', [])),
        'strengths'           : ' | '.join(audit.get('strengths', [])),
        'auditor_note'        : audit.get('auditor_note', ''),
        # FinBERT claim sub-scores
        'claim_sent_score'    : claim_sent_score,
        'claim_pos'           : claim_pos,
        'claim_neg'           : claim_neg,
        # Final Hybrid Talk Score
        'hybrid_talk_score'   : hybrid_talk
    })

    time.sleep(0.5)   # small pause between Groq API calls

df_talk = pd.DataFrame(talk_results)
df_talk.to_csv('talk_scores.csv', index=False)

print(f'\n{"="*60}')
print(' TALK SCORE SUMMARY')
print(f'{"="*60}')
print(f'Companies processed      : {len(df_talk)}')
print(f'Avg LLM Quality (all)    : {df_talk["llm_quality_score"].mean():.2f}/10')
print(f'Avg Hybrid Talk (all)    : {df_talk["hybrid_talk_score"].mean():.2f}/10')
print(f'Avg Talk (Verified)      : {df_talk[df_talk["bord_bia_verified"]=="True"]["hybrid_talk_score"].mean():.2f}/10')
print(f'\n Saved to talk_scores.csv')

---
## STAGE 3 — Walk Score (FinBERT on News: Headlines + Article Text)

**What is the Walk Score?**

If Talk = what a company *says*, Walk = what the *outside world says about them*.

For each company we fetch up to 5 Google News articles (headline + article body), run FinBERT on all of it, and convert the sentiment into a Walk Score (0–10).

```
Walk Score = 5 + (positive × 5) − (negative × 5)
```

> A score of 5 = neutral press. 8+ = strong positive coverage. Below 3 = negative press.

In [ ]:
# Define News Fetcher Functions


def fetch_article_body(url, max_chars=1000):
    """
    Visits a news article URL and extracts the first 1000 chars
    of paragraph text. Returns empty string on failure.

    ANALOGY: Instead of just reading the newspaper headline,
    we open the article and read the first few paragraphs too.
    """
    try:
        resp  = requests.get(url, headers=HEADERS, timeout=8, verify=False)
        soup  = BeautifulSoup(resp.text, 'html.parser')
        paras = soup.find_all('p')
        text  = ' '.join([p.get_text(strip=True) for p in paras[:5]])
        return text[:max_chars] if text else ''
    except Exception:
        return ''


def get_walk_score(company, max_articles=10):
    """
    For a given company:
      1. Fetch up to 5 Google News articles (RSS feed)
      2. Collect headline + article body text for each
      3. Run FinBERT across all collected texts
      4. Return Walk Score (0–10) + raw sentiment scores

    ANALOGY: Sending 5 journalists to investigate the company,
    reading their full articles, and averaging their verdict.
    """
    try:
        query = company['company_name'].replace(' ', '+')
        rss   = (f'https://news.google.com/rss/search'
                 f'?q={query}&hl=en-IE&gl=IE&ceid=IE:en')
        feed     = feedparser.parse(rss)
        articles = feed.entries[:max_articles]

        if not articles:
            return {
                'walk_positive' : 0.0,
                'walk_neutral'  : 1.0,
                'walk_negative' : 0.0,
                'walk_score'    : 5.0,   # neutral baseline when no news found
                'articles_used' : 0
            }

        all_texts = []
        for article in articles:
            # Always include the headline
            headline = article.get('title', '')
            if headline:
                all_texts.append(headline)
            # Also include the article body (more accurate)
            body = fetch_article_body(article.get('link', ''))
            if body:
                all_texts.append(body)

        scores     = analyse_sentiment(all_texts)
        walk_score = scores_to_sentiment_score(scores)

        return {
            'walk_positive' : scores['positive'],
            'walk_neutral'  : scores['neutral'],
            'walk_negative' : scores['negative'],
            'walk_score'    : walk_score,
            'articles_used' : len(articles)
        }

    except Exception:
        return {
            'walk_positive' : 0.0,
            'walk_neutral'  : 1.0,
            'walk_negative' : 0.0,
            'walk_score'    : 5.0,
            'articles_used' : 0
        }


print(' fetch_article_body() ready')
print(' get_walk_score() ready')

# --- Quick test on first company ---
print(f'\n Test walk score for: {companies[0]["company_name"]}')
w = get_walk_score(companies[0])
print(f'   Positive : {w["walk_positive"]}')
print(f'   Neutral  : {w["walk_neutral"]}')
print(f'   Negative : {w["walk_negative"]}')
print(f'   Walk Score: {w["walk_score"]} / 10')
print(f'   Articles : {w["articles_used"]}')

In [ ]:
# Run Walk Score on ALL Companies
# We fetch up to 10 texts per company (5 headlines + 5 article bodies)
# and run FinBERT on each one.

walk_results = []

print(f' Running Walk Score (FinBERT on news) for {len(companies)} companies')
print(f' Estimated time: 8–12 minutes\n')

for i, company in enumerate(companies, 1):
    name = company['company_name']
    print(f'[{i}/{len(companies)}] {name}...')

    result = get_walk_score(company)
    print(f'    Articles used: {result["articles_used"]} | Walk Score: {result["walk_score"]}/10')

    walk_results.append({
        'company_name'  : name,
        'walk_positive' : result['walk_positive'],
        'walk_neutral'  : result['walk_neutral'],
        'walk_negative' : result['walk_negative'],
        'walk_score'    : result['walk_score'],
        'articles_used' : result['articles_used']
    })

    time.sleep(1)   # respectful pause between companies

df_walk = pd.DataFrame(walk_results)
df_walk.to_csv('walk_scores.csv', index=False)

print(f'\n{"="*60}')
print(' WALK SCORE SUMMARY')
print(f'{"="*60}')
print(f'Companies processed    : {len(df_walk)}')
print(f'Avg Walk Score (all)   : {df_walk["walk_score"].mean():.2f}/10')
print(f'Avg Positive Sentiment : {df_walk["walk_positive"].mean():.4f}')
print(f'Avg Neutral Sentiment  : {df_walk["walk_neutral"].mean():.4f}')
print(f'Avg Negative Sentiment : {df_walk["walk_negative"].mean():.4f}')
print(f'\n--- Walk Scores per Company ---')
print(df_walk[['company_name', 'walk_positive', 'walk_negative', 'walk_score', 'articles_used']].to_string(index=False))
print(f'\n Saved to walk_scores.csv')

---
## STAGE 4 — Gap Analysis

**What is the Gap Score?**

The Gap Score measures how far a company's *claims* exceed their *public reality*.

```
Gap Score = max(0,  (Talk − Walk) / Talk )
```

- **0.0** = Talk and Walk are equal — claims match reality 
- **0.40+** = Talk is far higher than Walk — high greenwashing risk 

Before calculating the gap, we **normalise** both scores to the same 0–10 scale using min-max normalisation, so they're directly comparable.


In [ ]:
# Merge Talk + Walk, Normalise, Calculate Gap Score

def minmax_normalise(series):
    """
    Scales any numeric series to a 0–10 range.

    Formula: (value - min) / (max - min) * 10

    """
    mn, mx = series.min(), series.max()
    if mx == mn:
        return series * 0 + 5   # if all values identical, assign neutral 5
    return ((series - mn) / (mx - mn)) * 10


def calculate_gap_score(talk, walk):
    """
    Relative divergence between Talk and Walk.
    Returns 0 if talk=0 (can't have a gap with no claims).
    """
    if talk == 0:
        return 0.0
    return round(max(0.0, (talk - walk) / talk), 3)


def assign_risk_level(gap):
    """
    Converts a numeric gap score into a risk label.

    Thresholds:
      >= 0.40  →  HIGH RISK   (claims far exceed reality)
      >= 0.20  →  MEDIUM RISK (noticeable gap)
      >= 0.05  →  LOW RISK    (small gap)
      <  0.05  →  CREDIBLE    (claims match or understate reality)
    """
    if   gap >= 0.40: return '🔴 HIGH RISK'
    elif gap >= 0.20: return '🟡 MEDIUM RISK'
    elif gap >= 0.05: return '🟢 LOW RISK'
    else:             return '✅ CREDIBLE'

df_final = df_talk.merge(
    df_walk[['company_name', 'walk_positive', 'walk_neutral',
             'walk_negative', 'walk_score', 'articles_used']],
    on='company_name',
    how='left'
)

df_final['talk_norm'] = minmax_normalise(df_final['hybrid_talk_score']).round(2)
df_final['walk_norm'] = minmax_normalise(df_final['walk_score']).round(2)

df_final['gap_score']  = df_final.apply(
    lambda r: calculate_gap_score(r['talk_norm'], r['walk_norm']), axis=1
)
df_final['risk_level'] = df_final['gap_score'].apply(assign_risk_level)

df_final = df_final.sort_values('gap_score', ascending=False).reset_index(drop=True)

print(f'{"="*65}')
print(' GAP ANALYSIS RESULTS')
print(f'{"="*65}')
print(f'Companies analysed : {len(df_final)}')
print(f'🔴 High Risk       : {(df_final["risk_level"] == "🔴 HIGH RISK").sum()}')
print(f'🟡 Medium Risk     : {(df_final["risk_level"] == "🟡 MEDIUM RISK").sum()}')
print(f'🟢 Low Risk        : {(df_final["risk_level"] == "🟢 LOW RISK").sum()}')
print(f'✅ Credible        : {(df_final["risk_level"] == "✅ CREDIBLE").sum()}')

ver   = df_final[df_final['bord_bia_verified'] == True]
unver = df_final[df_final['bord_bia_verified'] == False]
print(f'\n📊 VERIFICATION COMPARISON')
print(f'Avg Gap — Bord Bia Verified : {ver["gap_score"].mean():.3f}')
print(f'Avg Gap — Not Verified      : {unver["gap_score"].mean():.3f}')
diff = unver['gap_score'].mean() - ver['gap_score'].mean()
pct  = round((diff / ver['gap_score'].mean()) * 100) if ver['gap_score'].mean() > 0 else 0
print(f'Difference                  : {diff:.3f}  ({pct}% higher for unverified)')

print(f'\n{"Company":<28} {"Talk":>6} {"Walk":>6} {"Gap":>7} {"Risk":<16} {"Group"}')
print('-' * 80)
for _, row in df_final.iterrows():
    print(f'{row["company_name"]:<28} '
          f'{row["talk_norm"]:>6.2f} '
          f'{row["walk_norm"]:>6.2f} '
          f'{row["gap_score"]:>7.3f} '
          f'{row["risk_level"]:<16} '
          f'{row["group"]}')

In [ ]:
# Sector-Level Gap Analysis

print(' SECTOR-LEVEL GAP ANALYSIS')
print('='*55)

sector_stats = df_final.groupby('sector').agg(
    companies     = ('company_name', 'count'),
    avg_talk      = ('talk_norm',    'mean'),
    avg_walk      = ('walk_norm',    'mean'),
    avg_gap       = ('gap_score',    'mean'),
    high_risk     = ('risk_level',   lambda x: (x == '🔴 HIGH RISK').sum()),
    verified_pct  = ('bord_bia_verified', lambda x: round(x.mean() * 100, 0))
).round(3).sort_values('avg_gap', ascending=False)

print(sector_stats.to_string())

print('\n🔺 Highest risk sector  :', sector_stats['avg_gap'].idxmax())
print('🟢 Lowest risk sector   :', sector_stats['avg_gap'].idxmin())

---
##  STAGE 5 — Recommendations

This stage generates **two types of recommendations**:
- **Per-company** — targeted action points based on each company's specific gap score, risk level, and audit findings
- **Sector-level** — patterns and systemic insights across food sectors

These are rule-based (no extra API calls needed) and export directly into the final CSV used by the dashboard.

In [ ]:
# Per-Company Recommendations

def generate_company_recommendation(row):
    """
    Generates a targeted recommendation string for one company.
    Uses: gap_score, risk_level, bord_bia_verified,
          talk_norm, walk_norm, greenwash_signals
    """
    recs = []
    gap  = row['gap_score']
    talk = row['talk_norm']
    walk = row['walk_norm']
    verified = row['bord_bia_verified']
    signals  = str(row.get('greenwash_signals', ''))

    # --- Primary risk recommendation ---
    if gap >= 0.40:
        recs.append(' HIGH PRIORITY: ESG claims significantly exceed public evidence. '
                    'Immediate independent verification strongly advised before any '
                    'ESG-linked investment or partnership.')
    elif gap >= 0.20:
        recs.append(' MODERATE ACTION: Gap between claims and public sentiment detected. '
                    'Recommend supplementing website claims with measurable data or '
                    'third-party certifications.')
    elif gap >= 0.05:
        recs.append(' LOW RISK: Minor gap detected. Consider publishing a formal '
                    'sustainability report with KPIs to close the credibility gap.')
    else:
        recs.append(' CREDIBLE PROFILE: Public reputation matches or exceeds ESG claims. '
                    'Maintain current approach and consider showcasing this credibility.')

    # --- Bord Bia verification recommendation ---
    if not verified and gap >= 0.20:
        recs.append(' Apply for Bord Bia Origin Green verification to provide '
                    'independent backing for ESG claims and reduce greenwashing risk.')
    elif verified and gap >= 0.30:
        recs.append(' Despite Bord Bia verification, gap score remains high. '
                    'Review public communications strategy — claims may be overstated.')

    # --- Talk score specific ---
    if talk >= 7.0 and walk <= 4.0:
        recs.append(' High claim volume but low public sentiment: consider whether '
                    'ESG messaging is reaching external audiences or is only internal-facing.')
    elif talk <= 3.0:
        recs.append(' Low ESG claim density on website: opportunity to better '
                    'communicate existing sustainability practices to customers and investors.')

    # --- Walk score specific ---
    if walk >= 7.0 and talk <= 4.0:
        recs.append(' Strong public reputation but under-claiming on website: '
                    'update website to reflect strong ESG track record.')

    # --- Greenwashing signal specific ---
    if 'vague' in signals.lower():
        recs.append(' LLM flagged vague language: replace generic phrases with '
                    'specific, measurable commitments (e.g., "30% emissions reduction by 2027").')
    if 'no measurable' in signals.lower() or 'no target' in signals.lower():
        recs.append(' No measurable targets detected: add quantified ESG KPIs '
                    'to sustainability page to improve credibility score.')
    if 'unverified' in signals.lower() or 'certification' in signals.lower():
        recs.append(' Unverified certifications flagged: ensure all mentioned '
                    'certifications are currently valid and publicly verifiable.')

    return ' | '.join(recs)


df_final['recommendation'] = df_final.apply(generate_company_recommendation, axis=1)

print(' Per-company recommendations generated')
print(f'\n--- Sample Recommendations ---')
for _, row in df_final.head(5).iterrows():
    print(f'\n{row["company_name"]} ({row["risk_level"]})')
    for r in row['recommendation'].split(' | '):
        print(f'   → {r}')

In [ ]:
# ============================================================
# CELL 5.2 — Sector-Level Recommendations
# ============================================================
# These are systemic insights across sectors — useful for
# your dissertation's findings section and dashboard.

print(' SECTOR-LEVEL RECOMMENDATIONS')
print('='*60)

for sector, grp in df_final.groupby('sector'):
    avg_gap  = grp['gap_score'].mean()
    avg_talk = grp['talk_norm'].mean()
    avg_walk = grp['walk_norm'].mean()
    n        = len(grp)
    verified = grp['bord_bia_verified'].sum()
    high_risk = (grp['risk_level'] == ' HIGH RISK').sum()

    print(f'\n Sector: {sector}  ({n} companies, {verified} verified)')
    print(f'   Avg Talk: {avg_talk:.2f} | Avg Walk: {avg_walk:.2f} | Avg Gap: {avg_gap:.3f}')

    if avg_gap >= 0.40:
        print(f'    HIGH RISK SECTOR: {high_risk}/{n} companies flagged as high risk.')
        print(f'      → Sector-wide audit recommended. Regulators and buyers should '
              f'apply heightened scrutiny to ESG claims in this sector.')
    elif avg_gap >= 0.20:
        print(f'    MODERATE RISK: Sector shows meaningful gap between claims and reputation.')
        print(f'      → Industry body (e.g. Bord Bia) could issue sector-specific '
              f'ESG disclosure guidelines to improve consistency.')
    else:
        print(f'    LOW RISK: Sector demonstrates credible ESG profile overall.')
        print(f'      → Good candidate for ESG best-practice showcase in Bord Bia '
              f'Origin Green programme communications.')

    if avg_talk > avg_walk + 2:
        print(f'     Claim-reality gap structural to sector — claims consistently '
              f'outpace news sentiment across all companies.')
    elif avg_walk > avg_talk + 2:
        print(f'    Under-claiming pattern — sector reputation exceeds stated claims. '
              f'Marketing opportunity for sector-wide ESG communication.')

In [ ]:
# ============================================================
# CELL 5.3 — Save Final Report CSV (feeds the Streamlit dashboard)
# ============================================================
# This is the SINGLE output CSV that your dashboard.py loads.
# It contains every column the dashboard needs.

OUTPUT_COLS = [
    # Company info
    'company_name', 'sector', 'county', 'group', 'bord_bia_verified',
    # Talk Score breakdown
    'claim_specificity', 'claim_credibility', 'llm_quality_score',
    'claim_sent_score', 'hybrid_talk_score', 'talk_norm',
    'claims_found', 'greenwash_signals', 'strengths', 'auditor_note',
    # Walk Score breakdown
    'walk_positive', 'walk_neutral', 'walk_negative',
    'walk_score', 'walk_norm', 'articles_used',
    # Gap + Risk
    'gap_score', 'risk_level',
    # Recommendations
    'recommendation'
]

# Keep only columns that exist (in case some are missing)
final_cols = [c for c in OUTPUT_COLS if c in df_final.columns]
df_output  = df_final[final_cols].copy()

# Rename columns to match what dashboard.py expects
df_output = df_output.rename(columns={
    'hybrid_talk_score' : 'talk_score',
    'walk_norm'         : 'walk_score_norm'
})

filename = 'final_greenwashing_report.csv'
df_output.to_csv(filename, index=False)

print(f'{"="*65}')
print('🌿 FINAL PIPELINE COMPLETE')
print(f'{"="*65}')
print(f'Companies in report  : {len(df_output)}')
print(f'Columns saved        : {len(df_output.columns)}')
print(f'Output file          : {filename}')
print(f'\n--- Columns in CSV ---')
for col in df_output.columns:
    print(f'   {col}')
print(f'\n--- Risk Distribution ---')
print(df_output['risk_level'].value_counts().to_string())
print(f'\n Done! Place {filename} in the same folder as dashboard.py and run:')
print('   streamlit run dashboard.py')

---
## Output Files Summary

| File | Contents | Used by |
|------|----------|--------|
| `talk_scores.csv` | LLM audit + FinBERT claim scores per company | Intermediate |
| `walk_scores.csv` | FinBERT news sentiment scores per company | Intermediate |
| `final_greenwashing_report.csv` | All scores + gap + risk + recommendations | **dashboard.py** |

---
##  Key Variable Reference

| Variable | Type | Description |
|----------|------|-------------|
| `df_talk` | DataFrame | Talk scores for all companies |
| `df_walk` | DataFrame | Walk scores for all companies |
| `df_final` | DataFrame | Merged + gap scored master table |
| `df_output` | DataFrame | Final export (matches dashboard column names) |
| `sector_stats` | DataFrame | Sector-level aggregated statistics |